In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-72B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-72B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
#     temperature=0.0,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|█████████████████████████████| 37/37 [01:49<00:00,  2.95s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [2]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "Qwen_72B_predictions_on_70B_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["70B_After_Removal"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        query_full = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Qwen_72B_predictions_70B.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['QA_ID', 'context', 'question_options', 'answer_df3', 'data_source_corr', 'Origin', '70B_sentence_ids', '70B_After_Removal', 'human_sentence_ids', 'Low_Irr_70B', 'Extra_Context_Minus_70B', 'gpt_direct_prediction', '72B_Sentence_Contents', '72B_Low_Irr']
Starting from scratch...
Processing 1297 rows...
Processing row 1/1297...
✅ Processed Merge Q1: Answer = D
Processing row 2/1297...
✅ Processed Merge Q2: Answer = C
Processing row 3/1297...
✅ Processed Merge Q3: Answer = D
Processing row 4/1297...
✅ Processed Merge Q4: Answer = C
Processing row 5/1297...
✅ Processed Merge Q5: Answer = D
Processing row 6/1297...
✅ Processed Merge Q6: Answer = D
Processing row 7/1297...
⚠️ Could not extract answer from response for row 7:
Response: To provide a clear and concise response, I will analyze the key features of the patient's presentati...
✅ Processed Merge Q7: Answer = None
Processing row 8/1297...
✅ Processed Merge Q8: Answer = B
Processing row 9/1297...
✅ Processed Merge

✅ Processed Merge Q109: Answer = D
Processing row 110/1297...
✅ Processed Merge Q110: Answer = B
Saved progress to CSV after 110 items
Processing row 111/1297...
✅ Processed Merge Q111: Answer = A
Processing row 112/1297...
✅ Processed Merge Q112: Answer = D
Processing row 113/1297...
✅ Processed Merge Q113: Answer = B
Processing row 114/1297...
✅ Processed Merge Q114: Answer = B
Processing row 115/1297...
✅ Processed Merge Q115: Answer = B
Processing row 116/1297...
✅ Processed Merge Q116: Answer = A
Processing row 117/1297...
✅ Processed Merge Q117: Answer = C
Processing row 118/1297...
✅ Processed Merge Q118: Answer = A
Processing row 119/1297...
✅ Processed Merge Q119: Answer = A
Processing row 120/1297...
✅ Processed Merge Q120: Answer = A
Saved progress to CSV after 120 items
Processing row 121/1297...
✅ Processed Merge Q121: Answer = B
Processing row 122/1297...
✅ Processed Merge Q122: Answer = A
Processing row 123/1297...
✅ Processed Merge Q123: Answer = C
Processing row 124/12

⚠️ Could not extract answer from response for row 219:
Response: To provide the best possible assistance, I would need to see Figure A (the ECG) to determine the mos...
✅ Processed Merge Q219: Answer = None
Processing row 220/1297...
✅ Processed Merge Q220: Answer = D
Saved progress to CSV after 220 items
Processing row 221/1297...
✅ Processed Merge Q221: Answer = C
Processing row 222/1297...
✅ Processed Merge Q222: Answer = C
Processing row 223/1297...
✅ Processed Merge Q223: Answer = B
Processing row 224/1297...
⚠️ Could not extract answer from response for row 224:
Response: To ensure you receive the correct response, please provide the full case summary as well.
To provide...
✅ Processed Merge Q224: Answer = None
Processing row 225/1297...
✅ Processed Merge Q225: Answer = D
Processing row 226/1297...
✅ Processed Merge Q226: Answer = B
Processing row 227/1297...
✅ Processed Merge Q227: Answer = C
Processing row 228/1297...
⚠️ Could not extract answer from response for row 228:
Respo

✅ Processed Merge Q316: Answer = B
Processing row 317/1297...
✅ Processed Merge Q317: Answer = H
Processing row 318/1297...
✅ Processed Merge Q318: Answer = I
Processing row 319/1297...
✅ Processed Merge Q319: Answer = J
Processing row 320/1297...
✅ Processed Merge Q320: Answer = G
Saved progress to CSV after 320 items
Processing row 321/1297...
✅ Processed Merge Q321: Answer = A
Processing row 322/1297...
✅ Processed Merge Q322: Answer = G
Processing row 323/1297...
✅ Processed Merge Q323: Answer = A
Processing row 324/1297...
✅ Processed Merge Q324: Answer = B
Processing row 325/1297...
✅ Processed Merge Q325: Answer = C
Processing row 326/1297...
✅ Processed Merge Q326: Answer = D
Processing row 327/1297...
✅ Processed Merge Q327: Answer = C
Processing row 328/1297...
✅ Processed Merge Q328: Answer = E
Processing row 329/1297...
⚠️ Could not extract answer from response for row 329:
Response: To diagnose the somatic dysfunction, we need to analyze the given information step by step:

✅ Processed Merge Q428: Answer = B
Processing row 429/1297...
✅ Processed Merge Q429: Answer = F
Processing row 430/1297...
✅ Processed Merge Q430: Answer = A
Saved progress to CSV after 430 items
Processing row 431/1297...
✅ Processed Merge Q431: Answer = D
Processing row 432/1297...
✅ Processed Merge Q432: Answer = G
Processing row 433/1297...
✅ Processed Merge Q433: Answer = J
Processing row 434/1297...
✅ Processed Merge Q434: Answer = D
Processing row 435/1297...
⚠️ Could not extract answer from response for row 435:
Response: To address the patient's condition, we need to identify the likely diagnosis and the medication used...
✅ Processed Merge Q435: Answer = None
Processing row 436/1297...
✅ Processed Merge Q436: Answer = C
Processing row 437/1297...
✅ Processed Merge Q437: Answer = A
Processing row 438/1297...
✅ Processed Merge Q438: Answer = I
Processing row 439/1297...
✅ Processed Merge Q439: Answer = J
Processing row 440/1297...
✅ Processed Merge Q440: Answer = C
Saved progr

...

✅ Processed Merge Q751: Answer = D
Processing row 752/1297...
✅ Processed Merge Q752: Answer = A
Processing row 753/1297...
✅ Processed Merge Q753: Answer = A
Processing row 754/1297...
✅ Processed Merge Q754: Answer = A
Processing row 755/1297...
✅ Processed Merge Q755: Answer = C
Processing row 756/1297...
✅ Processed Merge Q756: Answer = C
Processing row 757/1297...
✅ Processed Merge Q757: Answer = A
Processing row 758/1297...
✅ Processed Merge Q758: Answer = D
Processing row 759/1297...
✅ Processed Merge Q759: Answer = D
Processing row 760/1297...
✅ Processed Merge Q760: Answer = A
Saved progress to CSV after 760 items
Processing row 761/1297...
✅ Processed Merge Q761: Answer = C
Processing row 762/1297...
✅ Processed Merge Q762: Answer = C
Processing row 763/1297...
✅ Processed Merge Q763: Answer = D
Processing row 764/1297...
⚠️ Could not extract answer from response for row 764:
Response: To determine the most appropriate answer, let's analyze the key aspects of the study descri

✅ Processed Merge Q853: Answer = C
Processing row 854/1297...
✅ Processed Merge Q854: Answer = B
Processing row 855/1297...
✅ Processed Merge Q855: Answer = D
Processing row 856/1297...
✅ Processed Merge Q856: Answer = D
Processing row 857/1297...
✅ Processed Merge Q857: Answer = D
Processing row 858/1297...
✅ Processed Merge Q858: Answer = B
Processing row 859/1297...
✅ Processed Merge Q859: Answer = A
Processing row 860/1297...
✅ Processed Merge Q860: Answer = C
Saved progress to CSV after 860 items
Processing row 861/1297...
✅ Processed Merge Q861: Answer = D
Processing row 862/1297...
✅ Processed Merge Q862: Answer = D
Processing row 863/1297...
✅ Processed Merge Q863: Answer = B
Processing row 864/1297...
✅ Processed Merge Q864: Answer = D
Processing row 865/1297...
✅ Processed Merge Q865: Answer = A
Processing row 866/1297...
✅ Processed Merge Q866: Answer = B
Processing row 867/1297...
✅ Processed Merge Q867: Answer = B
Processing row 868/1297...
✅ Processed Merge Q868: Answer =

✅ Processed Merge Q965: Answer = B
Processing row 966/1297...
✅ Processed Merge Q966: Answer = D
Processing row 967/1297...
✅ Processed Merge Q967: Answer = D
Processing row 968/1297...
✅ Processed Merge Q968: Answer = C
Processing row 969/1297...
✅ Processed Merge Q969: Answer = D
Processing row 970/1297...
✅ Processed Merge Q970: Answer = C
Saved progress to CSV after 970 items
Processing row 971/1297...
✅ Processed Merge Q971: Answer = B
Processing row 972/1297...
✅ Processed Merge Q972: Answer = D
Processing row 973/1297...
✅ Processed Merge Q973: Answer = C
Processing row 974/1297...
✅ Processed Merge Q974: Answer = C
Processing row 975/1297...
✅ Processed Merge Q975: Answer = B
Processing row 976/1297...
✅ Processed Merge Q976: Answer = B
Processing row 977/1297...
✅ Processed Merge Q977: Answer = A
Processing row 978/1297...
✅ Processed Merge Q978: Answer = B
Processing row 979/1297...
✅ Processed Merge Q979: Answer = C
Processing row 980/1297...
✅ Processed Merge Q980: Answer =

✅ Processed Merge Q1080: Answer = C
Saved progress to CSV after 1080 items
Processing row 1081/1297...
✅ Processed Merge Q1081: Answer = D
Processing row 1082/1297...
✅ Processed Merge Q1082: Answer = B
Processing row 1083/1297...
✅ Processed Merge Q1083: Answer = C
Processing row 1084/1297...
✅ Processed Merge Q1084: Answer = A
Processing row 1085/1297...
✅ Processed Merge Q1085: Answer = D
Processing row 1086/1297...
✅ Processed Merge Q1086: Answer = A
Processing row 1087/1297...
✅ Processed Merge Q1087: Answer = C
Processing row 1088/1297...
✅ Processed Merge Q1088: Answer = C
Processing row 1089/1297...
✅ Processed Merge Q1089: Answer = D
Processing row 1090/1297...
✅ Processed Merge Q1090: Answer = B
Saved progress to CSV after 1090 items
Processing row 1091/1297...
✅ Processed Merge Q1091: Answer = B
Processing row 1092/1297...
✅ Processed Merge Q1092: Answer = C
Processing row 1093/1297...
✅ Processed Merge Q1093: Answer = C
Processing row 1094/1297...
✅ Processed Merge Q1094: A

✅ Processed Merge Q1201: Answer = B
Processing row 1202/1297...
✅ Processed Merge Q1202: Answer = B
Processing row 1203/1297...
✅ Processed Merge Q1203: Answer = C
Processing row 1204/1297...
✅ Processed Merge Q1204: Answer = C
Processing row 1205/1297...
✅ Processed Merge Q1205: Answer = D
Processing row 1206/1297...
✅ Processed Merge Q1206: Answer = B
Processing row 1207/1297...
✅ Processed Merge Q1207: Answer = D
Processing row 1208/1297...
✅ Processed Merge Q1208: Answer = B
Processing row 1209/1297...
✅ Processed Merge Q1209: Answer = C
Processing row 1210/1297...
✅ Processed Merge Q1210: Answer = C
Saved progress to CSV after 1210 items
Processing row 1211/1297...
✅ Processed Merge Q1211: Answer = B
Processing row 1212/1297...
✅ Processed Merge Q1212: Answer = D
Processing row 1213/1297...
✅ Processed Merge Q1213: Answer = C
Processing row 1214/1297...
✅ Processed Merge Q1214: Answer = D
Processing row 1215/1297...
✅ Processed Merge Q1215: Answer = B
Processing row 1216/1297...
✅

In [5]:
import pandas as pd
import re
import numpy as np
from scipy import stats

# Load the model predictions
output_df = pd.read_csv(paths.PREDICTIONS / "Qwen_72B_predictions_70B.csv")
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")

# Merge the two dataframes on ID_corr
merged_df = pd.merge(df, output_df[['Origin', 'Extracted_Answer']], on='Origin', how='inner')

# Compare answers
merged_df['72B_on_70B_Match'] = merged_df['answer_df3'] == merged_df['Extracted_Answer']
merged_df['72B_on_70B_Match'] = merged_df['72B_on_70B_Match'].map({True: 'TRUE', False: 'FALSE'})

# Exact match as 0/1 for statistics
merged_df['match'] = (merged_df['answer_df3'] == merged_df['Extracted_Answer']).astype(int)

# Overall accuracy statistics
accuracy = merged_df['match'].mean()
std_dev = merged_df['match'].std()
n = len(merged_df)
se = std_dev / np.sqrt(n)
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Category-wise analysis (if data_source_corr exists in df)
if 'data_source_corr' in df.columns:
    merged_df['data_source_corr'] = merged_df['data_source_corr']
    category_stats = merged_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()

    # Calculate 95% CI per category
    ci_lower, ci_upper = [], []
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100

    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()

# Summary table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.5875 (58.75%)
Standard Deviation: 0.4925
95% Confidence Interval: [0.5607, 0.6143]
95% CI (percentage): [56.07%, 61.43%]
Sample Size: 1297

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.647766 0.478077 0.019817     0.608845     0.686688        64.776632  47.807722      60.884476      68.668789
      medbullets    207       0.652174 0.477435 0.033184     0.586750     0.717598        65.217391  47.743511      58.675004      71.759778
        medxpert    315       0.279365 0.449401 0.025321     0.229545     0.329185        27.936508  44.940113      22.954503      32.918513
            mmlu    193       0.839378 0.368137 0.026499     0.787112     0.891645        83.937824  36.813667      78.711159      89.164489


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.5875 (58.75%)